In [1]:
import cutlass
import torch
from cutlass import cute
from cutlass.cute.runtime import from_dlpack
from cutlass.cute.nvgpu import cpasync

H_ORDERS = [4, 16, 64, 256, 1024, 4096]

class ReductionKernel():
    def __init__(self, h_order, reduction_mask):
        self.mask = reduction_mask
        self.h_order = H_ORDERS[h_order] 
        self.n_warps = self.h_order
        self.n_threads = self.h_order * 32

    @cute.jit
    def __call__(self, mX: cute.Tensor, mY: cute.Tensor):
        # Input:  X [B, H*W, C]
        # Output: Y [B, H*W // h_order, C]   (mean over h_order consecutive tokens)

        smem_layout = self.prepare_smem_layout(mX)
        print('smem_layout:', smem_layout)
        copy_atom   = self.prepare_copy_atom(mX)
        print('copy_atom:', copy_atom)

        # 128-bit copy = 8 float16 elements per transaction
        copy_elems = 128 // cutlass.Float16.width   # = 8
        # Thread layout: spread 32 threads across the (T, C) atom
        #   C_threads = threads along C  (each covers copy_elems columns)
        #   T_threads = remaining threads along T
        C_threads = 64 // copy_elems
        T_threads = 32 // C_threads
        tX_layout = cute.make_layout((T_threads, C_threads), stride=(C_threads, 1))
        vX_layout = cute.make_layout((1, copy_elems))
        gmem_tiled_copy = cute.make_tiled_copy_tv(copy_atom, tX_layout, vX_layout)
        print('gmem_tiled_copy:', gmem_tiled_copy)

        # Static shared-memory descriptor passed to the kernel
        @cute.struct
        class SharedStorage:
            sX: cute.struct.Align[
                cute.struct.MemRange[cutlass.Float16, cute.cosize(smem_layout)], 1024
            ]

        # One warp (32 threads) per output token; grid covers (B, HW_out)
        B  = mX.shape[0]
        HW = mX.shape[1]
        self.reduction_kernel(
            mX, mY, smem_layout, gmem_tiled_copy, SharedStorage,
        ).launch(
            grid=(B, HW // self.h_order),
            block=[32, 1, 1],
        )

    @cute.kernel
    def reduction_kernel(
        self,
        mX: cute.Tensor,
        mY: cute.Tensor,
        smem_layout: cute.ComposedLayout,
        gmem_tiled_copy: cute.TiledCopy,
        SharedStorage: cutlass.Constexpr,
    ):
        tidx, _, _ = cute.arch.thread_idx()
        bidx, bidy, _ = cute.arch.block_idx()   # unpack x, y, z

        # ------------------------------------------------------------------ #
        # 1. Global-memory tile: (h_order, C) starting at out_idx * h_order  #
        # ------------------------------------------------------------------ #
        b       = bidx      # batch index
        out_idx = bidy      # which output token this block produces

        # Slice mX[b] → (H*W, C), then extract the (h_order, C) tile
        print(mX.shape)
        gX = cute.local_tile(
            mX[b, None, None],
            (self.h_order, mX.shape[-1]),
            (out_idx, 0),
        )   # logical shape: (h_order, C)
        print(gX)

        smem = cutlass.utils.SmemAllocator()
        # # allocate smem storage
        storage = smem.allocate(SharedStorage)
        
        sX = storage.sX.get_tensor(smem_layout) # [h_order, C]
        
        gmem_thr_copy = gmem_tiled_copy.get_slice(tidx)

        tXgX = gmem_thr_copy.partition_S(gX)
        tXsX = gmem_thr_copy.partition_D(sX) 
        print('tXgX:', tXgX)
        print('tXsX:', tXsX)


        
        
        



        
        




    # ---------------------------------------------------------------------- #
    # Helpers                                                                  #
    # ---------------------------------------------------------------------- #

    def prepare_tensor(self, X: torch.Tensor) -> cute.Tensor:
        # Convert a contiguous [B, H*W, C] float16 torch tensor to a row-major
        # cute tensor.  assumed_align=16 guarantees 128-bit-aligned base ptr.
        return from_dlpack(X, assumed_align=16)

    def prepare_copy_atom(self, X: cute.Tensor):
        # Async G2S copy; 128 bits = 8 float16 elements per transaction
        return cute.make_copy_atom(
            cute.nvgpu.cpasync.CopyG2SOp(),
            cutlass.Float16,
            num_bits_per_copy=128,
        )

    def prepare_smem_layout(self, X: cute.Tensor):
        # Layout (C, T):  C = channels (major, with swizzle to avoid bank
        # conflicts during the vectorised gmem→smem copy), T = h_order (minor).
        # Atom mirrors the flash-attention smem atom: (c_block, 4) row-major
        # with a power-of-2 XOR swizzle matched to the copy width.
        C = X.shape[2]
        smem_c_block = 64 if C % 64 == 0 else 32
        swizzle_bits = 3  if smem_c_block == 64 else 2
        layout_atom  = cute.make_composed_layout(
            cute.make_swizzle(swizzle_bits, 3, 3),
            0,
            cute.make_layout((smem_c_block, 4), stride=(4, 1)),
        )
        # Tile the atom to cover the full (C, h_order) smem region
        return cute.tile_to_shape(layout_atom, (C, self.h_order), (0, 1))


In [2]:
# -----------------------------------------------------------------------
# Usage: always compile the @cute.jit callable first with cute.compile(),
# which establishes the MLIR context needed by the DSL.
# -----------------------------------------------------------------------

B, HW, C = 1, 4096, 64          # example: 1 batch, 256 tokens, 64 channels
h_order   = 0                   # reduce every 4 consecutive tokens → 1


X = torch.randn(B, HW, C, dtype=torch.float16, device="cuda")

Y = torch.zeros(B, HW // H_ORDERS[h_order], C, dtype=torch.float16, device="cuda")


kernel = ReductionKernel(h_order, reduction_mask=None)
X = kernel.prepare_tensor(X)
Y = kernel.prepare_tensor(Y)

# Build representative cute tensors so cute.compile can specialise on 

# cute.compile JIT-compiles the @cute.jit __call__ and returns a callable.
# Pass sample inputs so the compiler can infer all tensor types/shapes.
compiled_kernel = cute.compile(kernel, X, Y)

# Run the compiled kernel
compiled_kernel( X, Y)



smem_layout: S<3,3,3> o 0 o ((64,1),(4,1)):((4,0),(1,0))
copy_atom: Copy Atom
  ThrID:         1:0
  TV Layout Src: (1,8):(0,1)
  TV Layout Dst: (1,8):(0,1)
  Value type:    f16
gmem_tiled_copy: Tiled Copy
  Tiler MN:        (4:1,64:1)
  TV Layout tiled: ((8,4),8):((32,1),4)
Copy Atom
  ThrID:           1:0
  TV Layout Src:   (1,8):(0,1)
  TV Layout Dst:   (1,8):(0,1)
  Value type:      f16
(1, 4096, 64)
tensor<ptr<f16, gmem> o (4,64):(64,1)>
tXgX: tensor<ptr<f16, gmem> o ((8,1),1,1):((1,0),0,0)>
tXsX: tensor<ptr<f16, smem, align<8>> o S<3,3,3> o ?{div=8} o (((4,2),1),16,1):(((1,0),0),16,0)>


0